In [1]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = str(PROJECT_ROOT / "data/raw/pdf/2026_hope_ladder_selected.pdf")
docs = PyPDFLoader(file_path=file_path).load()
print(f"페이지 수: {len(docs)}")

c:\Users\user\catcher-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


페이지 수: 49


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("rag_index")
print(f"완료: {vectorstore.index.ntotal}개 벡터 저장")

완료: 49개 벡터 저장


In [4]:
vectorstore = FAISS.load_local("rag_index", embeddings, allow_dangerous_deserialization=True)

In [5]:
retriever = vectorstore.as_retriever()
query = "임산부 지원 정책"

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):
    print(f"[{i}위] p.{doc.metadata['page']+1}")
    print(doc.page_content[:300])
    print("=" * 60)

[1위] p.21
모두의	정책	K-희망사다리	2026 121
1588-2188
정부24
맘편한 임신 원스톱 서비스
지원대상 	 •	 신청일	기준	임산부
핵심내용 	 •	 임신	후	받을	수	있는	각종	임신지원	서비스를	한	번에	안내받고	통합	신청하는	
서비스
	 •전국	공통	서비스
	 •지자체	서비스
		 ※		지방자치단체별로	제공	내역이	다르므로	관할	읍·면·동	행정복지센터에	확인	
	 •	서비스별	처리	기관에서	문자	또는	유선으로	개별	안내	및	서비스	제공
		 ※		택배	신청	시	엽산·철분제는	건강기능식품	제공,	택배요금은	임산부	부담
	 	 ※
[2위] p.19
모두의	정책	K-희망사다리	2026 119
월 1회 이상
임산부·영유아	대상	
영양지원	서비스
129
보건복지상담센터
임산부 및 영유아 
영양플러스
지원대상 	 •	 임신부,	출산	및	수유부,	5세	이하	영·유아
	 •	가구	규모별	기준	중위소득	80%	이하	
	 •	영양위험요인	보유자(빈혈,	저체중,	성장부진,	영양섭취상태	불량	등	한	가지	
이상	영양위험	보유)
핵심내용 	
이용방법 	 •	 방문	신청:	거주	지역	보건소
문의처	 •거주	지역	보건소,	보건복지상담센터(☎129)
구분    내용
영양교육 및 상담
	최소	월	1회	
[3위] p.26
128생애주기별 국민생활 서비스 - 가족·여성
1577-4206
가족상담전화
한부모가족 복지시설
지원대상 	 •	 18세	미만	아동을	양육하는	무주택	저소득	한부모가족	
	 •	소득인정액	기준	중위소득	100%	이하
	 •	소득	무관	입소:	①	위기	임산부,	②	임신	중이거나	출산	후	1년	이내	해당하는	
한부모(출산지원시설),	③	인구감소지역에	설치·운영	중인	한부모가족	복지
시설	
핵심내용 	 •	 전국	116개소	운영	중(2026년	기준)
이용방법 	 •	 방문	신청:	해당	복지시설,	시·군·구	한부모가족복지시설	담당	부서
문
[4위] p.2
043모두의 정책 K-희망사다리 2026
129
보건복지상담센터
임신 사전건강관리 
지원사업
지원대상 	